## 1. Configuration

`--tau 0.01` is the paper value; **the CLI default is `0.1`**, ten times higher. `--plddt` stays
off: it costs 6.91&nbsp;GB VRAM and pulls a 1.6B model regardless of checkpoint (`inference.py:152`).

In [ ]:
import os, sys, json, time, subprocess, glob, shutil, collections
import numpy as np

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, f'targets_vhh.json: expected 1, found {_ds}'
DATASET = _ds[0]
FORK    = 'https://github.com/omar-A-hassan/ml-simplefold'
BRANCH  = 'fix-multichain-fasta-parsing'   # multichain FASTA fix (apple/ml-simplefold PR #59)

MODEL, TAU, STEPS, SEED = 'simplefold_100M', '0.01', '500', '12345'

WORK, SCRATCH = '/kaggle/working', '/tmp/sf'   # working is 20GB; /tmp has ~1TB
os.makedirs(SCRATCH, exist_ok=True)
T0 = time.time()
R = {'model': MODEL,
     'settings': {'tau': float(TAU), 'num_steps': int(STEPS), 'seed': int(SEED),
                  'n_samples': 1, 'plddt': False},
     'metric': 'ImmuneBuilder protocol: IMGT regions, N/CA/C/CB atoms',
     'reference_only': {'NanoBodyBuilder2': {'cdr1': 1.98, 'cdr2': 1.37, 'cdr3': 2.89,
                                             'framework': 0.79},
                        'AlphaFold2_cdr3': 3.44,
                        'caveat': 'ImmuneBuilder Tab.3 -- DIFFERENT test set, not paired'}}

def note(tag):
    R['elapsed_min']  = round((time.time() - T0) / 60, 1)
    R['stage']        = tag
    R['work_free_gb'] = round(shutil.disk_usage(WORK).free / 2**30, 1)
    R['tmp_free_gb']  = round(shutil.disk_usage('/tmp').free / 2**30, 1)
    json.dump(R, open(f'{WORK}/results_vhh.json', 'w'), indent=2, default=str)
    print(f"[{R['elapsed_min']:>6.1f} min] {tag}   /kaggle/working {R['work_free_gb']}GB free",
          flush=True)

def sh(c):
    p = subprocess.run(c, shell=True, text=True, capture_output=True)
    return p

def must(c, what):
    """Run a shell command and fail LOUDLY with its output, not a bare AssertionError."""
    p = sh(c)
    if p.returncode != 0:
        raise RuntimeError(f'{what} failed (rc={p.returncode}):\n{(p.stdout + p.stderr)[-2000:]}')
    return p

import torch
n, cap = torch.cuda.device_count(), torch.cuda.get_device_capability(0)
R['gpus'] = [torch.cuda.get_device_name(g) for g in range(n)]
assert n == 2, f'need T4 x2, got {n}: {R["gpus"]}'
assert cap >= (7, 0), f'{R["gpus"][0]} is sm_{cap[0]}{cap[1]}; this torch build needs sm_70+'
print('GPUs:', R['gpus'])
note('start')

## 2. Install SimpleFold from the fork branch



In [ ]:
must(f'git clone -q --depth 1 -b {BRANCH} {FORK} {SCRATCH}/repo', 'git clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo', 'pip install simplefold')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git', 'pip install esm')
must(f'{sys.executable} -m pip install -q biotite scipy', 'pip install biotite/scipy')

src = open(f'{SCRATCH}/repo/src/simplefold/utils/fasta_utils.py').read()
assert 'chr(ord("A") + i)' in src, 'multichain fix absent -- wrong branch?'
print('multichain FASTA fix present')
note('installed')

## 3. Patch ESM to fp16 

In [ ]:
INF = f'{SCRATCH}/repo/src/simplefold/inference.py'
s = open(INF).read()
anchor = '        esm_model = esm_model.to(device)'
assert s.count(anchor) == 1, f'anchor found {s.count(anchor)}x -- upstream changed'
open(INF, 'w').write(s.replace(anchor, '        esm_model = esm_model.half().to(device)'))
R['patch'] = 'esm .half() applied in-notebook'
print('ESM2-3B will load fp16 (~11GB -> ~5.6GB)')
note('patched')

## 4. Load data



In [ ]:
TG = json.load(open(DATASET))
for t in TG:
    assert t['atoms'] == ['N','CA','C','CB'], (t['instance'], t['atoms'])
    assert t['numbering'] == 'IMGT'
    assert len(t['hca']) == len(t['hseq'])
    for k in ('cdr1','cdr2','cdr3'):
        lo, hi = t[k]
        assert 0 <= lo < hi <= len(t['hseq'])
    # spans must reproduce SAbDab2's own CDR strings
    for k, col in (('cdr1','CDRH1'), ('cdr2','CDRH2'), ('cdr3','CDRH3')):
        assert t['hseq'][t[k][0]:t[k][1]] == t[col], (t['instance'], k)
assert len({t['cdrh3_cluster'] for t in TG}) == len(TG), 'not one target per CDRH3 cluster'
assert len({t['instance'] for t in TG}) == len(TG), 'duplicate instance ids'
R['n_targets'] = len(TG)

# FASTA stem becomes the prediction filename, so it must be unique and collision-free
tid_of = {t['instance']: f"{t['pdb']}{t['H']}" for t in TG}
tids = list(tid_of.values())
assert len(set(tids)) == len(tids), 'duplicate FASTA stems'
assert not [1 for a in set(tids) for b in set(tids) if a != b and a in b], \
       'one stem is a substring of another -- glob would match the wrong structure'

fadir = f'{SCRATCH}/fa'; os.makedirs(fadir, exist_ok=True)
for t in TG:
    tid = tid_of[t['instance']]
    open(f'{fadir}/{tid}.fasta', 'w').write(f">A|{tid}\n{t['hseq']}\n")

L  = [len(t['hseq']) for t in TG]
c3 = [t['cdr3'][1] - t['cdr3'][0] for t in TG]
print(f'{len(TG)} nanobodies | length median {int(np.median(L))} ({min(L)}-{max(L)})'
      f' | CDR-H3 median {int(np.median(c3))} ({min(c3)}-{max(c3)})')
print(' method :', dict(collections.Counter(t['method'] for t in TG)))
print(' holo   :', dict(collections.Counter(t['holo'] for t in TG)))
print(' split  :', dict(collections.Counter(t['ab_split'] for t in TG)))
note('targets')

## 5. Warm the caches single-threaded



In [ ]:
must(f'{sys.executable} -c "import torch, esm; '
     f'torch.hub.load(\'facebookresearch/esm:main\', \'esm2_t36_3B_UR50D\')"', 'torch.hub warm')

warm = f'{SCRATCH}/warm'; os.makedirs(warm, exist_ok=True)
open(f'{warm}/w.fasta','w').write(f">A|w\n{TG[0]['hseq']}\n")
w = subprocess.run(['simplefold','--simplefold_model',MODEL,'--backend','torch','--ckpt_dir',f'{SCRATCH}/artifacts',
                    '--num_steps','10','--tau',TAU,'--nsample_per_protein','1','--seed','1',
                    '--fasta_path',warm,'--output_dir',f'{SCRATCH}/warmout'],
                   capture_output=True, text=True,
                   env={**os.environ, 'CUDA_VISIBLE_DEVICES':'0'})
R['warm_rc'] = w.returncode
if w.returncode:
    raise RuntimeError('checkpoint warm-up failed; aborting rather than racing:\n'
                       + (w.stdout + w.stderr)[-2500:])
print('checkpoints warm')
note('warmed')

## 6. Fold on both T4s



In [ ]:
import torch
assert torch.cuda.device_count() == 2, \
    f'need T4 x2, got {torch.cuda.device_count()} -- set the accelerator before running'

for g in (0, 1):
    sh(f'rm -rf {SCRATCH}/sh{g} {SCRATCH}/out{g}')   # idempotent on re-run
    os.makedirs(f'{SCRATCH}/sh{g}', exist_ok=True)
for n, f in enumerate(sorted(glob.glob(f'{fadir}/*.fasta'))):
    os.link(f, f'{SCRATCH}/sh{n % 2}/{os.path.basename(f)}')
R['shard_sizes'] = [len(os.listdir(f'{SCRATCH}/sh{g}')) for g in (0, 1)]
assert sum(R['shard_sizes']) == len(TG) and min(R['shard_sizes']) > 0, R['shard_sizes']
print('shard sizes:', R['shard_sizes'])

t0 = time.time()
procs = []
for g in (0, 1):
    if g:
        time.sleep(90)
    procs.append(subprocess.Popen(
        ['simplefold','--simplefold_model',MODEL,'--backend','torch','--ckpt_dir',f'{SCRATCH}/artifacts','--num_steps',STEPS,
         '--tau',TAU,'--nsample_per_protein','1','--seed',SEED,
         '--fasta_path',f'{SCRATCH}/sh{g}','--output_dir',f'{SCRATCH}/out{g}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        env={**os.environ,'CUDA_VISIBLE_DEVICES':str(g),
             'PYTORCH_ALLOC_CONF':'expandable_segments:True'}))
    print(f'launched GPU {g}', flush=True)

for g, p in enumerate(procs):
    out = p.communicate()[0]
    R.setdefault('fold_rc', {})[f'gpu{g}'] = p.returncode
    if p.returncode:
        R.setdefault('fold_tail', {})[f'gpu{g}'] = out[-2000:]
        print(f'--- GPU {g} FAILED (rc={p.returncode}) ---\n', out[-2000:])
R['fold_sec'] = round(time.time() - t0, 1)     # includes the 90s stagger
R['n_predicted'] = len(glob.glob(f'{SCRATCH}/out*/**/*.cif', recursive=True))
R['sec_per_target'] = round(R['fold_sec'] / max(R['n_predicted'], 1), 2)
print(f"{R['n_predicted']}/{len(TG)} predicted in {R['fold_sec']/60:.1f} min"
      f"  ({R['sec_per_target']}s/target incl. stagger)")
if any(R['fold_rc'].values()):
    print('\nWARNING: a GPU worker failed. Scoring will proceed on whatever exists,')
    print('but the missing targets are NOT random -- check fold_tail before trusting medians.')
note('folded')

## 7. Score



In [ ]:
import biotite.structure.io.pdbx as pdbx
from scipy import stats

def kabsch(P, Q):
    Pm, Qm = P.mean(0), Q.mean(0)
    V, _, W = np.linalg.svd((P - Pm).T @ (Q - Qm))
    D = np.diag([1, 1, np.sign(np.linalg.det(V @ W))])
    return V @ D @ W, Pm, Qm

rmsd = lambda a, b: float(np.sqrt(((a - b) ** 2).sum(-1).mean()))
REGIONS = ['cdr1', 'cdr2', 'cdr3', 'framework']

rows = []
for t in TG:
    tid = tid_of[t['instance']]
    try:
        g = glob.glob(f'{SCRATCH}/out*/**/*{tid}*.cif', recursive=True)
        if not g:
            R.setdefault('missing', []).append(t['instance']); continue
        if len(g) > 1:
            R.setdefault('ambiguous', []).append(f'{tid}: {len(g)} files'); continue
        a  = pdbx.get_structure(pdbx.CIFFile.read(g[0]), model=1)
        Ln = len(t['hseq'])
        # prediction must be ONE chain numbered contiguously over exactly Ln residues, else
        # res_id - min() silently misaligns every residue and still yields plausible RMSDs
        u = np.unique(a.res_id)
        if len(np.unique(a.chain_id)) != 1 or len(u) != Ln or u.max() - u.min() + 1 != Ln:
            R.setdefault('bad_numbering', []).append(
                f"{t['instance']}: chains={len(np.unique(a.chain_id))} nres={len(u)} want={Ln}")
            continue
        pred = np.full((Ln, 4, 3), np.nan)
        rid  = a.res_id - a.res_id.min()
        for k, an in enumerate(['N', 'CA', 'C', 'CB']):
            m = a.atom_name == an
            pred[rid[m], k] = a.coord[m]
        ref = np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']], float)

        reg = {}
        for k in ('cdr1', 'cdr2', 'cdr3'):
            m = np.zeros(Ln, bool); lo, hi = t[k]; m[lo:hi] = True; reg[k] = m
        reg['framework'] = ~(reg['cdr1'] | reg['cdr2'] | reg['cdr3'])

        ok = np.isfinite(ref[..., 0]) & np.isfinite(pred[..., 0])
        if ok[:, 1].sum() < 60:
            R.setdefault('skipped', []).append(f"{t['instance']}:coverage"); continue

        row = {'instance': t['instance'], 'tid': tid, 'L': Ln,
               'cdr3_len': int(t['cdr3'][1] - t['cdr3'][0]),
               'n_atoms_matched': int(ok.sum()),
               'method': t['method'], 'holo': t['holo'], 'ab_split': t['ab_split'],
               'resolution': t['resolution']}
        for mode, fitmask in (('chain', np.ones(Ln, bool)), ('fw', reg['framework'])):
            sel = ok & fitmask[:, None]
            if sel.sum() < 60:
                continue
            Rot, Pm, Qm = kabsch(pred[sel], ref[sel])
            al = np.full_like(pred, np.nan)
            al[ok] = (pred[ok] - Pm) @ Rot + Qm
            for rn in REGIONS:
                s2 = ok & reg[rn][:, None]
                row[f'{mode}_{rn}'] = round(rmsd(al[s2], ref[s2]), 3) if s2.sum() >= 3 else None
        rows.append(row)
    except Exception as e:
        R.setdefault('score_err', []).append(f"{t['instance']}:{type(e).__name__}:{str(e)[:90]}")

R['rows'] = rows
lost = len(TG) - len(rows)
print(f'scored {len(rows)} / {len(TG)}   (unscored {lost})')
for k in ('missing','ambiguous','bad_numbering','skipped','score_err'):
    if R.get(k): print(f'  {k}: {len(R[k])}  e.g. {R[k][:2]}')
assert lost < 0.05 * len(TG), f'{lost} targets unscored -- too many to trust the medians'
note('scored')

## 7b. Per-target scores

In [ ]:
import csv
cols = ['instance','tid','L','cdr3_len','n_atoms_matched','method','holo','ab_split',
        'resolution'] + [f'{m}_{r}' for m in ('chain','fw') for r in REGIONS]
json.dump(rows, open(f'{WORK}/per_target_scores.json', 'w'), indent=1)
with open(f'{WORK}/per_target_scores.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction='ignore', restval='')
    w.writeheader(); w.writerows(rows)
print(f'wrote per_target_scores.json/.csv  ({len(rows)} rows, {len(cols)} cols)')

## 8. Free the predictions

In [ ]:
for g in (0, 1):
    sh(f'rm -rf {SCRATCH}/out{g}')
print('freed;', round(shutil.disk_usage('/tmp').free / 2**30, 1), 'GB free in /tmp')
print('working dir holds only JSON/CSV -- output pull stays small')

## 9. Results



In [ ]:
def summarise(rr):
    rng = np.random.default_rng(0)
    out = {'n': len(rr)}
    for mode in ('chain', 'fw'):
        d = {}
        for rn in REGIONS:
            v = np.array([r[f'{mode}_{rn}'] for r in rr if r.get(f'{mode}_{rn}') is not None])
            if len(v) >= 5:
                bs = np.median(v[rng.integers(0, len(v), (2000, len(v)))], axis=1)
                d[rn] = {'median': round(float(np.median(v)), 3),
                         'mean':   round(float(v.mean()), 3),
                         'ci95':  [round(float(np.percentile(bs, 2.5)), 3),
                                   round(float(np.percentile(bs, 97.5)), 3)],
                         'n': int(len(v))}
        out[f'{mode}_aligned'] = d
    return out

S = summarise(rows)
nbb = R['reference_only']['NanoBodyBuilder2']
ca  = S['chain_aligned']
print(f"chain-aligned (ImmuneBuilder protocol), n={S['n']}\n")
print(f"{'region':<11}{'median':>9}{'mean':>9}{'95% CI':>18}   {'| NBB2 (other set)':>18}")
for rn in REGIONS:
    if rn not in ca: continue
    e = ca[rn]
    ci = f"[{e['ci95'][0]}, {e['ci95'][1]}]"
    print(f"{rn:<11}{e['median']:>9.3f}{e['mean']:>9.3f}{ci:>18}   "
          f"{'| ' + str(nbb[rn]):>18}")
print('\nfw-aligned (isolates loop error):')
for rn in REGIONS:
    e = S['fw_aligned'].get(rn)
    if e: print(f"  {rn:<11}{e['median']:>9.3f}")
print('\nNBB2 = ImmuneBuilder Tab.3, a DIFFERENT test set. Interpretable side by side,')
print('not a controlled comparison. No delta reported.')

# ---- stratification required by METRICS.md
strata = {'ab_split==test': [r for r in rows if r['ab_split'] == 'test'],
          'method==XRAY':   [r for r in rows if r['method'] == 'XRAY'],
          'method==EM':     [r for r in rows if r['method'] == 'EM'],
          'holo==True':     [r for r in rows if str(r['holo']) == 'True'],
          'holo==False':    [r for r in rows if str(r['holo']) == 'False']}
cl = np.array([r['cdr3_len'] for r in rows])
for lo, hi in ((0, 13), (14, 17), (18, 99)):
    strata[f'cdr3_len {lo}-{hi}'] = [r for r in rows if lo <= r['cdr3_len'] <= hi]
print(f"\n{'stratum':<18}{'n':>5}{'cdr3 median':>13}{'95% CI':>18}")
S['strata'] = {}
for k, rr in strata.items():
    if len(rr) < 5: continue
    ss = summarise(rr); e = ss['chain_aligned'].get('cdr3')
    if not e: continue
    S['strata'][k] = ss
    ci = f"[{e['ci95'][0]}, {e['ci95'][1]}]"
    print(f"{k:<18}{len(rr):>5}{e['median']:>13.3f}{ci:>18}")

x = np.array([r['cdr3_len'] for r in rows], float)
y = np.array([r['fw_cdr3'] if r.get('fw_cdr3') is not None else np.nan for r in rows])
m = np.isfinite(y)
if m.sum() > 20:
    S['cdr3_err_vs_len_spearman'] = round(float(stats.spearmanr(x[m], y[m]).statistic), 3)
    print(f"\nCDR-H3 error vs CDR-H3 length: spearman {S['cdr3_err_vs_len_spearman']}")

S['reference_NanoBodyBuilder2'] = dict(R['reference_only'])
R['summary'] = S
note('done')
print('\nwrote /kaggle/working/results_vhh.json')